# Extract CoM seed point from Segmentation Masks

The FMCIB `get_features` function expects image paths and seed points. If you have segmentation masks you would like to convert to CoM, this notebook shows how.

Alternatively, you can use the Mhub [fmcib_radiomics](https://mhub.ai/models/fmcib_radiomics) implementation and the `nrrd_mask_workflow`.

In [ ]:
import SimpleITK as sitk
from fmcib.utils import download_LUNG1, build_image_seed_dict
from fmcib.visualization import visualize_seed_point

First, we download a sample from LUNG1 to show centroid extraction. Use your own data here and skip this step.

The download and conversion will take about a minute.

In [ ]:
download_LUNG1("dummy", samples=1)
build_image_seed_dict("dummy")

Now we get the path to the image and mask. This can be nii.gz, nrrd, mha or other formats supported by MONAI's ITKReader.

In [ ]:
import pathlib

dummy_path = pathlib.Path("dummy")
image_path = list(dummy_path.rglob("image.nii.gz"))[0]
mask_path = list(dummy_path.rglob("*GTV-1.nii.gz"))[0]

In [ ]:
row = {"image_path": image_path, "label_path": mask_path}

The visualize seed point utility also visualizes masks when `label_path` is provided as a key in the dict.

In [ ]:
visualize_seed_point(row)

We can now convert the label to centroid coordinates in physical space:

1. Load the mask using SimpleITK
2. Use SimpleITK `LabelShapeStatisticsImageFilter` to get the centroid

In [ ]:
mask = sitk.ReadImage(mask_path)

label_shape_filter = sitk.LabelShapeStatisticsImageFilter()
label_shape_filter.Execute(mask)
try:
    centroid = label_shape_filter.GetCentroid(255)
except Exception:
    centroid = label_shape_filter.GetCentroid(1)

x, y, z = centroid

coordinate_dict = {
    "coordX": x,
    "coordY": y,
    "coordZ": z,
}

In [ ]:
coordinate_dict

In [ ]:
row = {"image_path": image_path, **coordinate_dict}

Now call the visualize seed point function again, this time with `image_path` and coordinate values.

In [ ]:
visualize_seed_point(row)

The bounding box that will be passed to FMCIB, centered around the seed point, is now shown.

---

# Full-lesion resize for WORC datasets

FMCIB's default path resamples to **1 mm** and crops a **50 mm** cube around the seed. That truncates large WORC lesions.

This section instead:

1. Builds a **cubic** window that fully contains each lesion (bbox center + max side + 4 mm margin), **never smaller than 50 mm**
2. Resamples that cube to **50×50×50** (aspect ratio preserved; voxel size varies per lesion)
3. Writes resized NIfTIs under `data/fmcib_full_lesion/`
4. Writes one CSV per dataset under `metadata/fmcib_full_lesion_{Dataset}.csv`

CSV columns: `image_path`, `coordX`, `coordY`, `coordZ`, `label`. Paths are **repo-relative** (e.g. `data/fmcib_full_lesion/CRLM/...`) so they work on Colab if the notebook cwd is the repo root.

Feature extraction **must** use `precropped=True`, otherwise FMCIB will resample to 1 mm and recrop 50 mm:

```python
from fmcib.run import get_features
df = get_features("metadata/fmcib_full_lesion_CRLM.csv", precropped=True)
```

Note: FMCIB still applies a CT HU window (`-1024` / `3072`) even for MRI cohorts (Lipo, Desmoid, Liver).

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk

REPO_ROOT = Path.cwd().resolve()
WORC_ROOT = REPO_ROOT / "data" / "worc"
OUT_CROP_ROOT = REPO_ROOT / "data" / "fmcib_full_lesion"
OUT_CSV_ROOT = REPO_ROOT / "metadata"
PINFO_ROOT = REPO_ROOT / "metadata"

DATASETS = ("Lipo", "Desmoid", "Liver", "GIST", "CRLM", "Melanoma")
EXPECTED_LESIONS = {
    "Lipo": 116,
    "Desmoid": 203,
    "Liver": 186,
    "GIST": 247,
    "CRLM": 93,
    "Melanoma": 169,
}
OUT_SIZE = (50, 50, 50)
MARGIN_MM = 4.0
# Never smaller than FMCIB's published 50 mm window; expand only for larger lesions.
MIN_SIDE_MM = 50.0


def get_foreground_label(stats: sitk.LabelShapeStatisticsImageFilter) -> int:
    labels = list(stats.GetLabels())
    if not labels:
        raise ValueError("Mask has no foreground labels")
    if 255 in labels:
        return 255
    if 1 in labels:
        return 1
    return int(labels[0])


def get_centroid(mask: sitk.Image) -> tuple[float, float, float]:
    stats = sitk.LabelShapeStatisticsImageFilter()
    stats.Execute(mask)
    label = get_foreground_label(stats)
    return tuple(float(x) for x in stats.GetCentroid(label))


def bbox_center_and_extents_mm(mask: sitk.Image) -> tuple[tuple[float, float, float], np.ndarray]:
    """Axis-aligned bbox center (physical mm) and extents along image axes (mm)."""
    stats = sitk.LabelShapeStatisticsImageFilter()
    stats.Execute(mask)
    label = get_foreground_label(stats)
    x, y, z, sx, sy, sz = stats.GetBoundingBox(label)
    spacing = np.array(mask.GetSpacing(), dtype=float)
    extents = np.array([sx, sy, sz], dtype=float) * spacing
    center_idx = (x + sx / 2.0, y + sy / 2.0, z + sz / 2.0)
    center_phys = mask.TransformContinuousIndexToPhysicalPoint(center_idx)
    return tuple(float(c) for c in center_phys), extents


def resample_full_lesion_cube(
    image: sitk.Image,
    mask: sitk.Image,
    out_size: tuple[int, int, int] = OUT_SIZE,
    margin_mm: float = MARGIN_MM,
) -> tuple[sitk.Image, tuple[float, float, float], float, float, tuple[float, float, float]]:
    """
    Crop a cube that fully contains the mask and resample to out_size^3.

    Returns: crop, bbox_center_phys, side_mm, voxel_spacing_mm, com_phys
    """
    center_phys, extents = bbox_center_and_extents_mm(mask)
    com_phys = get_centroid(mask)
    side_mm = float(max(np.max(extents) + margin_mm, MIN_SIDE_MM))
    voxel_spacing = side_mm / out_size[0]

    direction = np.array(image.GetDirection(), dtype=float).reshape(3, 3)
    new_spacing = np.array([voxel_spacing, voxel_spacing, voxel_spacing], dtype=float)
    center_cont = np.array([(s - 1) / 2.0 for s in out_size], dtype=float)
    new_origin = tuple(np.array(center_phys, dtype=float) - direction @ (center_cont * new_spacing))

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(tuple(float(s) for s in new_spacing))
    resampler.SetSize(out_size)
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetOutputOrigin(new_origin)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0.0)
    crop = resampler.Execute(image)
    return crop, center_phys, side_mm, voxel_spacing, com_phys


def load_pinfo(dataset: str) -> dict[str, int]:
    path = PINFO_ROOT / f"pinfo_{dataset}.csv"
    df = pd.read_csv(path)
    return {str(row["Patient"]): int(row["Diagnosis"]) for _, row in df.iterrows()}


def resolve_label(dataset: str, patient_id: str, lesion_id: str, pinfo: dict[str, int]) -> int:
    if dataset == "Lipo" and lesion_id == "WDLPS":
        return 1
    if dataset == "Lipo" and lesion_id == "Lipoma":
        return 0
    if patient_id not in pinfo:
        raise KeyError(f"No pinfo label for {patient_id}")
    return pinfo[patient_id]


def discover_lesion_masks(dataset: str, patient_dir: Path) -> list[tuple[str, Path, str | None]]:
    """
    Return list of (lesion_id, mask_path, segmentation_source).
    CRLM: prefer RAD, fallback STUD1.
    """
    if dataset == "Lipo":
        wdlps = patient_dir / "segmentation_WDLPS.nii.gz"
        lipoma = patient_dir / "segmentation_Lipoma.nii.gz"
        if wdlps.exists() and lipoma.exists():
            return [("WDLPS", wdlps, None), ("Lipoma", lipoma, None)]

    if dataset == "CRLM":
        by_lesion: dict[str, dict[str, Path]] = {}
        for path in sorted(patient_dir.glob("segmentation_lesion*_*.nii.gz")):
            m = re.match(r"segmentation_(lesion\d+)_(RAD|STUD1|STUD2|PhD|CNN)\.nii\.gz$", path.name)
            if not m:
                continue
            lesion_id, source = m.group(1), m.group(2)
            by_lesion.setdefault(lesion_id, {})[source] = path
        rows = []
        for lesion_id, sources in sorted(by_lesion.items()):
            if "RAD" in sources:
                rows.append((lesion_id, sources["RAD"], "RAD"))
            elif "STUD1" in sources:
                rows.append((lesion_id, sources["STUD1"], "STUD1"))
            else:
                raise FileNotFoundError(f"No RAD/STUD1 mask for {patient_dir.name} {lesion_id}")
        return rows

    # Melanoma: segmentation_lesion0.nii.gz
    melanoma_masks = sorted(patient_dir.glob("segmentation_lesion*.nii.gz"))
    melanoma_masks = [p for p in melanoma_masks if re.match(r"segmentation_lesion\d+\.nii\.gz$", p.name)]
    if melanoma_masks:
        return [
            (re.match(r"segmentation_(lesion\d+)\.nii\.gz$", p.name).group(1), p, None)
            for p in melanoma_masks
        ]

    # GIST dual: segmentation_lesion_0.nii.gz
    gist_masks = sorted(patient_dir.glob("segmentation_lesion_*.nii.gz"))
    if gist_masks:
        return [
            (re.match(r"segmentation_(lesion_\d+)\.nii\.gz$", p.name).group(1), p, None)
            for p in gist_masks
        ]

    default = patient_dir / "segmentation.nii.gz"
    if default.exists():
        return [("lesion0", default, None)]

    raise FileNotFoundError(f"No segmentation found in {patient_dir}")


print("Helpers ready.")
print("WORC_ROOT exists:", WORC_ROOT.exists())
print("Datasets present:", [d for d in DATASETS if (WORC_ROOT / d).is_dir()])

In [ ]:
def resolve_image_path(patient_dir: Path, lesion_id: str) -> Path:
    """Prefer shared image.nii.gz; fall back to per-lesion image_{lesion_id}.nii.gz (e.g. GIST-018)."""
    shared = patient_dir / "image.nii.gz"
    if shared.exists():
        return shared
    per_lesion = patient_dir / f"image_{lesion_id}.nii.gz"
    if per_lesion.exists():
        return per_lesion
    raise FileNotFoundError(
        f"Missing image for {patient_dir.name} lesion {lesion_id}: "
        f"tried {shared.name} and {per_lesion.name}"
    )


def process_dataset(dataset: str) -> pd.DataFrame:
    pinfo = load_pinfo(dataset)
    patient_dirs = sorted(p for p in (WORC_ROOT / dataset).iterdir() if p.is_dir())
    out_dir = OUT_CROP_ROOT / dataset
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    qc_rows = []
    missing_rad = 0
    for patient_dir in patient_dirs:
        patient_id = patient_dir.name
        lesions = discover_lesion_masks(dataset, patient_dir)
        image_cache: dict[Path, sitk.Image] = {}

        for lesion_id, mask_path, seg_source in lesions:
            if dataset == "CRLM" and seg_source == "STUD1":
                missing_rad += 1

            image_path = resolve_image_path(patient_dir, lesion_id)
            if image_path not in image_cache:
                image_cache[image_path] = sitk.ReadImage(str(image_path))
            image = image_cache[image_path]

            mask = sitk.ReadImage(str(mask_path))
            crop, center_phys, side_mm, voxel_spacing, com_phys = resample_full_lesion_cube(image, mask)
            _, extents = bbox_center_and_extents_mm(mask)

            out_name = f"{patient_id}__{lesion_id}.nii.gz"
            out_path = out_dir / out_name
            sitk.WriteImage(crop, str(out_path))

            label = resolve_label(dataset, patient_id, lesion_id, pinfo)
            rows.append(
                {
                    "image_path": out_path.relative_to(REPO_ROOT).as_posix(),
                    "coordX": center_phys[0],
                    "coordY": center_phys[1],
                    "coordZ": center_phys[2],
                    "label": int(label),
                }
            )
            qc_rows.append(
                {
                    "crop_side_mm": side_mm,
                    "voxel_spacing_mm": voxel_spacing,
                    "bbox_max_extent_mm": float(np.max(extents)),
                    "exceeds_50mm_published_crop": bool(np.max(extents) > 50.0),
                }
            )

    df = pd.DataFrame(rows)
    qc = pd.DataFrame(qc_rows)
    csv_path = OUT_CSV_ROOT / f"fmcib_full_lesion_{dataset}.csv"
    df.to_csv(csv_path, index=False)

    expected = EXPECTED_LESIONS[dataset]
    n_over_50 = int(qc["exceeds_50mm_published_crop"].sum())
    print(f"\n=== {dataset} ===")
    print(f"Wrote {len(df)} rows -> {csv_path}")
    print(f"Expected lesions: {expected} | delta: {len(df) - expected}")
    if dataset == "CRLM":
        print(f"CRLM lesions using STUD1 fallback (no RAD): {missing_rad}")
    print(
        "crop_side_mm min/median/max:",
        f"{qc['crop_side_mm'].min():.1f} / {qc['crop_side_mm'].median():.1f} / {qc['crop_side_mm'].max():.1f}",
    )
    print(
        "voxel_spacing_mm min/median/max:",
        f"{qc['voxel_spacing_mm'].min():.3f} / {qc['voxel_spacing_mm'].median():.3f} / {qc['voxel_spacing_mm'].max():.3f}",
    )
    print(f"Lesions with bbox max extent > 50 mm (would be truncated by published FMCIB crop): {n_over_50}/{len(df)}")
    return df


all_dfs = {}
for ds in DATASETS:
    all_dfs[ds] = process_dataset(ds)

print("\nDone. Use get_features(..., precropped=True) on these CSVs.")

Alternatively, run the same pipeline from the shell (recommended for the full cohort):

```bash
python scripts/build_fmcib_full_lesion_csvs.py
```

Then extract features with:

```python
from fmcib.run import get_features

features = get_features("metadata/fmcib_full_lesion_CRLM.csv", precropped=True)
```

Do **not** call `get_features` with `precropped=False` on these CSVs: that would resample to 1 mm and recrop a 50 mm window, undoing the full-lesion resize.